# Portfolio Assignment 2 (M4): Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL

## 🧱 Part A: Baseline Model (Frozen Embeddings)

In [1]:
# Import and Install libraries and packages
%pip install datasets transformers sentence-transformers scikit-learn torch -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# loading the Dataset: AI-Growth-Lab/patents_claims_1.5m_traim_test
from datasets import load_dataset

dataset = load_dataset("AI-Growth-Lab/patents_claims_1.5m_traim_test")

/home/roger/Documents/Business Data Science AAU25/2semester/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset.keys()

dict_keys(['train', 'test'])

In [4]:
len(dataset["train"])

1372910

In [5]:
len(dataset["train"].column_names)

666

In [6]:
(len(dataset["train"]), len(dataset["train"].column_names))

(1372910, 666)

In [7]:
dataset["train"].features

{'id': Value('int64'),
 'date': Value('string'),
 'text': Value('string'),
 'A01B': Value('int64'),
 'A01C': Value('int64'),
 'A01D': Value('int64'),
 'A01F': Value('int64'),
 'A01G': Value('int64'),
 'A01H': Value('int64'),
 'A01J': Value('int64'),
 'A01K': Value('int64'),
 'A01L': Value('int64'),
 'A01M': Value('int64'),
 'A01N': Value('int64'),
 'A21B': Value('int64'),
 'A21C': Value('int64'),
 'A21D': Value('int64'),
 'A22B': Value('int64'),
 'A22C': Value('int64'),
 'A23B': Value('int64'),
 'A23C': Value('int64'),
 'A23D': Value('int64'),
 'A23F': Value('int64'),
 'A23G': Value('int64'),
 'A23J': Value('int64'),
 'A23K': Value('int64'),
 'A23L': Value('int64'),
 'A23N': Value('int64'),
 'A23P': Value('int64'),
 'A23V': Value('int64'),
 'A23Y': Value('int64'),
 'A24B': Value('int64'),
 'A24C': Value('int64'),
 'A24D': Value('int64'),
 'A24F': Value('int64'),
 'A41B': Value('int64'),
 'A41C': Value('int64'),
 'A41D': Value('int64'),
 'A41F': Value('int64'),
 'A41G': Value('int64'),


In [8]:
dataset["train"][:5]

{'id': [8788730, 8621421, 9461433, 9229528, 8508147],
 'date': ['2014-07-22',
  '2013-12-31',
  '2016-10-04',
  '2016-01-05',
  '2013-08-13'],
 'text': ['1. A method for sending a keycode of a non-keyboard apparatus, comprising the steps of: (a) connecting the non-keyboard apparatus to a computer so as to perform device enumeration and generate enumeration information, wherein the enumeration information is recorded by the non-keyboard apparatus and includes an enumeration value; (b) identifying, according to the enumeration value, the kind of an operating system used by the computer, and recording the kind of the operating system by the non-keyboard apparatus; and (c) reading the kind of the operating system so as to determine a preset second keycode that matches the kind of the operating system, wherein the second keycode is an ASCII (American Standard Code for Information Interchange) code.',
  '1. A method executed at least in part in a computing device for providing workflow visua

In [9]:
len(dataset["test"])

119384

In [10]:
len(dataset["test"].column_names)

666

In [11]:
(len(dataset["test"]), len(dataset["test"].column_names))

(119384, 666)

In [12]:
dataset["test"].features

{'id': Value('int64'),
 'date': Value('string'),
 'text': Value('string'),
 'A01B': Value('int64'),
 'A01C': Value('int64'),
 'A01D': Value('int64'),
 'A01F': Value('int64'),
 'A01G': Value('int64'),
 'A01H': Value('int64'),
 'A01J': Value('int64'),
 'A01K': Value('int64'),
 'A01L': Value('int64'),
 'A01M': Value('int64'),
 'A01N': Value('int64'),
 'A21B': Value('int64'),
 'A21C': Value('int64'),
 'A21D': Value('int64'),
 'A22B': Value('int64'),
 'A22C': Value('int64'),
 'A23B': Value('int64'),
 'A23C': Value('int64'),
 'A23D': Value('int64'),
 'A23F': Value('int64'),
 'A23G': Value('int64'),
 'A23J': Value('int64'),
 'A23K': Value('int64'),
 'A23L': Value('int64'),
 'A23N': Value('int64'),
 'A23P': Value('int64'),
 'A23V': Value('int64'),
 'A23Y': Value('int64'),
 'A24B': Value('int64'),
 'A24C': Value('int64'),
 'A24D': Value('int64'),
 'A24F': Value('int64'),
 'A41B': Value('int64'),
 'A41C': Value('int64'),
 'A41D': Value('int64'),
 'A41F': Value('int64'),
 'A41G': Value('int64'),


In [13]:
dataset["test"][:5]

{'id': [9289350, 9579560, 9269063, 9576958, 8441548],
 'date': ['2016-03-22',
  '2017-02-28',
  '2016-02-23',
  '2017-02-21',
  '2013-05-14'],
 'text': ['1. An apparatus for generating air pressure and high frequency air pulses to a garment having an air core located adjacent the body of a person whereby the body of the person is subjected to pressure and high frequency pulses, the apparatus comprising: a generator for producing air pressure and high frequency air pulses adapted to be operatively connected to the garment to transmit air pressure and high frequency pulses to the air core of the garment, an electric motor drivably connected to the generator for operating the generator to produce air pressure and high frequency air pressure pulses, a valve for restricting the flow of air to the generator to regulate the air pressure produced by the generator, said valve including an air flow control member for regulating the flow of air to the generator, a control device operably connecte

In [14]:
y02_cols = [col for col in dataset["train"].column_names if col.startswith("Y02")]
y02_cols

['Y02A', 'Y02B', 'Y02C', 'Y02D', 'Y02E', 'Y02P', 'Y02T', 'Y02W']

In [15]:
def is_green(example):
    return any(example[col] == 1 for col in y02_cols)

green_count = dataset["train"].filter(is_green).num_rows
green_count

103692

In [16]:
green_count/len(dataset["train"])

0.07552716492705276

In [17]:
# splitting dataset: green and non-green
train = dataset["train"]

green = train.filter(is_green)
not_green = train.filter(lambda x: not is_green(x))

In [18]:
green_25k = green.shuffle(seed=42).select(range(25000))
not_green_25k = not_green.shuffle(seed=42).select(range(25000))

In [19]:
from datasets import concatenate_datasets

balanced_50k = concatenate_datasets([green_25k, not_green_25k])
balanced_50k = balanced_50k.shuffle(seed=42)

In [20]:
len(balanced_50k)

50000

In [ ]:
balanced_50k.to_parquet("patents_50k_green.parquet")

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

### The steps above, have been executed in Collabs.google, due to physical constraints on my laptop

In [2]:
from datasets import load_dataset

dataset_50k = load_dataset("parquet", data_files="patents_50k_green.parquet")

In [3]:
data = dataset_50k["train"]

In [4]:
len(data)

50000

In [5]:
data = data.shuffle(seed=42)

In [6]:
train_silver = data.select(range(30000))
eval_silver = data.select(range(30000, 40000))
pool_unlabeled = data.select(range(40000, 50000))

In [7]:
from datasets import DatasetDict

final_dataset = DatasetDict({
    "train_silver": train_silver,
    "eval_silver": eval_silver,
    "pool_unlabeled": pool_unlabeled
})

In [8]:
final_dataset.save_to_disk("patents_50k_green_splits")

Saving the dataset (1/1 shards): 100%|██████████| 10000/10000 [00:17<00:00, 563.19 examples/s]


In [9]:
from datasets import load_from_disk

dataset = load_from_disk("patents_50k_green_splits")
train_silver = dataset["train_silver"]
eval_silver = dataset["eval_silver"]

In [10]:
len(train_silver), len(eval_silver)

(30000, 10000)

Load PatentSBERTa

In [11]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "AI-Growth-Lab/PatentSBERTa"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

for param in model.parameters():
    param.requires_grad = False

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 745.54it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: AI-Growth-Lab/PatentSBERTa
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings function

In [12]:
y02_cols = [col for col in train_silver.column_names if col.startswith("Y02")]

In [13]:
def add_green_label(example):
    example["is_green_silver"] = int(
        any(example[col] == 1 for col in y02_cols)
    )
    return example

train_silver = train_silver.map(add_green_label)
eval_silver = eval_silver.map(add_green_label)

In [14]:
"is_green_silver" in train_silver.column_names

True

In [15]:
from tqdm import tqdm
import numpy as np

In [16]:
def get_embeddings(texts, batch_size=32):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        # NÃO mover para CUDA
        with torch.no_grad():
            outputs = model(**inputs)

        last_hidden = outputs.last_hidden_state
        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        summed = torch.sum(last_hidden * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts

        embeddings.append(mean_pooled.numpy())

    return np.vstack(embeddings)

In [ ]:
X_train = get_embeddings(train_silver["text"], batch_size=32)
np.save("X_train.npy", X_train)


  0%|          | 1/938 [00:15<3:57:33, 15.21s/it]


KeyboardInterrupt: 

In [17]:
X_train = np.load("X_train.npy")

In [ ]:
X_eval = get_embeddings(eval_silver["text"], batch_size=64)
np.save("X_eval.npy", X_eval)

100%|██████████| 157/157 [7:26:50<00:00, 170.77s/it]    


In [18]:
X_eval = np.load("X_eval.npy")

In [19]:
y_train = np.array(train_silver["is_green_silver"])
y_eval = np.array(eval_silver["is_green_silver"])

In [20]:
# Training Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [21]:
# Evaluation
from sklearn.metrics import classification_report

y_pred = clf.predict(X_eval)

print(classification_report(y_eval, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.78      0.78      5063
           1       0.78      0.77      0.77      4937

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.78     10000
weighted avg       0.78      0.78      0.78     10000



## 🎯 Part B: Identify High-Risk Examples (Uncertainty Sampling)

In [ ]:
X_pool = get_embeddings(pool_unlabeled["text"], batch_size=32)
np.save("X_pool.npy", X_pool)

  0%|          | 0/313 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [22]:
X_pool = np.load("X_pool.npy")

In [23]:
p_green = clf.predict_proba(X_pool)[:, 1]

In [24]:
u = 1 - 2 * np.abs(p_green - 0.5)

In [25]:
top_indices = np.argsort(-u)[:100]

In [26]:
top_indices

array([5010, 9807, 8151, 9363, 8168, 1981, 1890, 8283, 9595, 8716, 7866,
       9527, 7064, 2013,  132, 1241, 3408, 8484, 3144, 6667,  277, 7626,
       2273,  564, 2270, 9299, 8707, 5461, 1276, 4436, 7390, 7051, 9169,
       5109, 9373, 6609, 8084, 9229, 3653, 6231, 5162, 7511, 8043, 3784,
       5217, 3724, 8530, 8039, 2287, 1871,  467, 9124, 8930, 4573, 1151,
       1673, 5047, 3504, 4900, 3014, 6133, 6585, 6508, 7211,  730, 7142,
       1595,  761, 3815, 3644, 9364, 2105, 1980, 1621, 8275, 4151, 9859,
       6829, 6384, 3266, 7091, 5378, 9537, 2580, 8731, 3822, 3167, 2554,
         82, 9389, 3856, 5797, 7891, 9881,  476, 8300,  788, 6229, 9826,
       8525])

In [27]:
import pandas as pd

df_hitl = pd.DataFrame({
    "doc_id": pool_unlabeled.select(top_indices)["id"],
    "text": pool_unlabeled.select(top_indices)["text"],
    "p_green": p_green[top_indices],
    "u": u[top_indices],
    "llm_green_suggested": "",
    "llm_confidence": "",
    "llm_rationale": "",
    "is_green_human": ""
})

df_hitl.to_csv("hitl_green_100.csv", index=False)

## 🤝 Part C: Implement LLM → Human HITL (Gold Labels)

In [28]:
# Load the 100 high-risk examples selected in Part B
hitl_df = pd.read_csv("hitl_green_100.csv")

print("HITL size:", len(hitl_df))

HITL size: 100


In [29]:
# Generate embeddings only for the 100 HITL examples
X_hitl = get_embeddings(hitl_df["text"].tolist(), batch_size=32)
np.save("X_hitl.npy", X_hitl)

100%|██████████| 4/4 [00:23<00:00,  5.96s/it]


In [30]:
X_hitl = np.load("X_hitl.npy")

In [31]:
# Get predicted probabilities from baseline classifier
probs = clf.predict_proba(X_hitl)

# Probability of class 1 (green)
p_green = probs[:, 1]

# LLM suggestion (threshold 0.5)
hitl_df["llm_green_suggested"] = (p_green > 0.5).astype(int)

# Confidence based on distance from 0.5
hitl_df["llm_confidence"] = np.where(
    np.abs(p_green - 0.5) > 0.25,
    "high",
    np.where(np.abs(p_green - 0.5) > 0.1, "medium", "low")
)

# Simple explanation field (student-level acceptable)
hitl_df["llm_rationale"] = (
    "Prediction based on PatentSBERTa semantic embedding similarity "
    "to green vs non-green training claims."
)

In [32]:
# Create empty human label column
hitl_df["is_green_human"] = ""

# Optional notes column
hitl_df["notes"] = ""

In [33]:
hitl_df.to_csv("hitl_review_ready.tsv", sep="\t", index=False)

In [34]:
import numpy as np
print(np.min(p_green), np.max(p_green))

0.4926092631004322 0.5072448587580276


In [35]:
p_eval = clf.predict_proba(X_eval)[:, 1]
print(np.min(p_eval), np.max(p_eval))

0.003689741149113534 0.9985175691876433


## 🚀 Part D: Final Model (Fine-Tune PatentSBERTa Once)

### Step 1 — Merge Gold Labels

In [36]:
train_df = train_silver.to_pandas()
eval_df = eval_silver.to_pandas()

In [37]:
hitl_df = pd.read_csv("hitl_reviewed.csv", sep="\t")

In [38]:
print(train_silver.column_names)

['id', 'date', 'text', 'A01B', 'A01C', 'A01D', 'A01F', 'A01G', 'A01H', 'A01J', 'A01K', 'A01L', 'A01M', 'A01N', 'A21B', 'A21C', 'A21D', 'A22B', 'A22C', 'A23B', 'A23C', 'A23D', 'A23F', 'A23G', 'A23J', 'A23K', 'A23L', 'A23N', 'A23P', 'A23V', 'A23Y', 'A24B', 'A24C', 'A24D', 'A24F', 'A41B', 'A41C', 'A41D', 'A41F', 'A41G', 'A41H', 'A42B', 'A42C', 'A43B', 'A43C', 'A43D', 'A44B', 'A44C', 'A44D', 'A45B', 'A45C', 'A45D', 'A45F', 'A46B', 'A46D', 'A47B', 'A47C', 'A47D', 'A47F', 'A47G', 'A47H', 'A47J', 'A47K', 'A47L', 'A61B', 'A61C', 'A61D', 'A61F', 'A61G', 'A61H', 'A61J', 'A61K', 'A61L', 'A61M', 'A61N', 'A61P', 'A61Q', 'A62B', 'A62C', 'A62D', 'A63B', 'A63C', 'A63D', 'A63F', 'A63G', 'A63H', 'A63J', 'A63K', 'B01B', 'B01D', 'B01F', 'B01J', 'B01L', 'B02B', 'B02C', 'B03B', 'B03C', 'B03D', 'B04B', 'B04C', 'B05B', 'B05C', 'B05D', 'B06B', 'B07B', 'B07C', 'B08B', 'B09B', 'B09C', 'B21B', 'B21C', 'B21D', 'B21F', 'B21G', 'B21H', 'B21J', 'B21K', 'B21L', 'B22C', 'B22D', 'B22F', 'B23B', 'B23C', 'B23D', 'B23F', '

In [39]:
gold_dict = dict(zip(hitl_df["doc_id"], hitl_df["is_green_human"]))

In [40]:
def override_label(example):
    patent_id = example["id"]

    if patent_id in gold_dict:
        example["is_green_gold"] = gold_dict[patent_id]
    else:
        example["is_green_gold"] = example["is_green_silver"]

    return example

train_gold = train_silver.map(override_label)

In [41]:
print(train_gold.column_names)

['id', 'date', 'text', 'A01B', 'A01C', 'A01D', 'A01F', 'A01G', 'A01H', 'A01J', 'A01K', 'A01L', 'A01M', 'A01N', 'A21B', 'A21C', 'A21D', 'A22B', 'A22C', 'A23B', 'A23C', 'A23D', 'A23F', 'A23G', 'A23J', 'A23K', 'A23L', 'A23N', 'A23P', 'A23V', 'A23Y', 'A24B', 'A24C', 'A24D', 'A24F', 'A41B', 'A41C', 'A41D', 'A41F', 'A41G', 'A41H', 'A42B', 'A42C', 'A43B', 'A43C', 'A43D', 'A44B', 'A44C', 'A44D', 'A45B', 'A45C', 'A45D', 'A45F', 'A46B', 'A46D', 'A47B', 'A47C', 'A47D', 'A47F', 'A47G', 'A47H', 'A47J', 'A47K', 'A47L', 'A61B', 'A61C', 'A61D', 'A61F', 'A61G', 'A61H', 'A61J', 'A61K', 'A61L', 'A61M', 'A61N', 'A61P', 'A61Q', 'A62B', 'A62C', 'A62D', 'A63B', 'A63C', 'A63D', 'A63F', 'A63G', 'A63H', 'A63J', 'A63K', 'B01B', 'B01D', 'B01F', 'B01J', 'B01L', 'B02B', 'B02C', 'B03B', 'B03C', 'B03D', 'B04B', 'B04C', 'B05B', 'B05C', 'B05D', 'B06B', 'B07B', 'B07C', 'B08B', 'B09B', 'B09C', 'B21B', 'B21C', 'B21D', 'B21F', 'B21G', 'B21H', 'B21J', 'B21K', 'B21L', 'B22C', 'B22D', 'B22F', 'B23B', 'B23C', 'B23D', 'B23F', '

In [42]:
from datasets import Dataset

train_final = train_gold.remove_columns(
    [col for col in train_gold.column_names if col not in ["text", "is_green_gold"]]
)

eval_final = eval_silver.remove_columns(
    [col for col in eval_silver.column_names if col not in ["text", "is_green_silver"]]
)

train_final = train_final.rename_column("is_green_gold", "label")
eval_final = eval_final.rename_column("is_green_silver", "label")

In [43]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "AI-Growth-Lab/PatentSBERTa"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 760.37it/s, Materializing param=mpnet.encoder.relative_attention_bias.weight]     
MPNetForSequenceClassification LOAD REPORT from: AI-Growth-Lab/PatentSBERTa
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [44]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_final = train_final.map(tokenize, batched=True)
eval_final = eval_final.map(tokenize, batched=True)

train_final.set_format("torch")
eval_final.set_format("torch")

Map: 100%|██████████| 10000/10000 [00:02<00:00, 4184.44 examples/s]


In [45]:
import accelerate
print(accelerate.__version__)

1.12.0


In [46]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./patentsberta_final",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [47]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_final,
    eval_dataset=eval_final,
    compute_metrics=compute_metrics
)

trainer.train()

/ceph/home/student.aau.dk/vf36ha/aau-codes/2sem/M4-DLML/assignments/ass2/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
eval_results = trainer.evaluate()
print("Evaluation on eval_silver:")
print(eval_results)

In [ ]:
from datasets import Dataset

gold_dataset = Dataset.from_pandas(
    hitl_df[["text", "is_green_human"]]
)

gold_dataset = gold_dataset.rename_column("is_green_human", "label")
gold_dataset = gold_dataset.map(tokenize, batched=True)
gold_dataset.set_format("torch")

gold_results = trainer.evaluate(gold_dataset)

print("Evaluation on gold_100:")
print(gold_results)